# Same-String Answerability Causal Pilot

This notebook executes the frozen task-local causal pilot. It does not alter the closed behavioral or representation results. A causal claim is permitted only after both fresh test splits and every registered control are complete.

In [ ]:
from pathlib import Path
import os, subprocess, sys

PINNED_REPO_COMMIT = "a69c773774f4bbb2d1994c060e41ee85ed007b28"
CHECKOUT = Path("/content/answerability-familiarity")
if not CHECKOUT.exists():
    subprocess.run(["git", "clone", "https://github.com/Fredo220/Answerability-x-Familarity-.git", str(CHECKOUT)], check=True)
subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "origin", PINNED_REPO_COMMIT], check=True)
subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", PINNED_REPO_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(CHECKOUT / "requirements/fa-causal-colab.lock")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(CHECKOUT)], check=True)
sys.path.insert(0, str(CHECKOUT / "src"))
if not os.environ.get("HF_TOKEN"):
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
assert os.environ.get("HF_TOKEN"), "Add HF_TOKEN through Colab Secrets before continuing."

In [ ]:
USE_DRIVE_CHECKPOINTS = True
ARTIFACT_ROOT = Path("/content/fa-causal-pilot-v1")
if USE_DRIVE_CHECKPOINTS:
    try:
        from google.colab import drive
        drive.mount("/content/drive", timeout_ms=60_000)
        ARTIFACT_ROOT = Path("/content/drive/MyDrive/fa-causal-pilot-v1")
    except (TimeoutError, ValueError) as error:
        print(f"Drive unavailable; using local checkpoints: {error}")
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print(ARTIFACT_ROOT)

In [ ]:
import argparse, json, torch
from trajectory_extractor.fa_answerability_causal_cli import (
    CausalDependencies, CausalTokenizerBinding, HFCausalRunner,
    load_causal_config, prepare_causal, run_causal_validation,
    expected_causal_shards, run_causal_shard, evaluate_causal,
)

assert torch.cuda.is_available(), "Select a Colab GPU runtime."
CONFIG = CHECKOUT / "configs/familiarity_answerability_causal_pilot_v1.json"
V3_CORPUS = CHECKOUT / "release/familiarity_answerability/representation_replication_v3/same_string_replication_v3_manifest.json"
V3_ACTIVATIONS = CHECKOUT / "release/familiarity_answerability/representation_replication_v3/activations/activations-representation_train.manifest.json"
config = load_causal_config(CONFIG)

In [ ]:
# Preparation uses only the pinned tokenizer and the closed v3 training artifacts.
prepared = prepare_causal(argparse.Namespace(
    config=str(CONFIG), root=str(ARTIFACT_ROOT),
    v3_corpus_manifest=str(V3_CORPUS),
    v3_training_activation_manifest=str(V3_ACTIVATIONS),
    output_dir="prepared",
))
print(json.dumps(prepared, indent=2))

In [ ]:
# Load Gemma once. All later calls reuse this exact instance.
runner = HFCausalRunner.from_pretrained(config)
binding = CausalTokenizerBinding(
    tokenizer=runner.tokenizer, model_id=config.model_id,
    model_revision=config.model_revision, tokenizer_id=config.model_id,
    tokenizer_revision=config.tokenizer_revision,
    chat_template_sha256=config.chat_template_sha256,
)
deps = CausalDependencies(
    tokenizer_loader=lambda _config: binding,
    runner_factory=lambda _config: runner,
)

In [ ]:
validation = run_causal_validation(argparse.Namespace(
    config=str(CONFIG), root=str(ARTIFACT_ROOT),
    prepare_manifest=prepared["prepare_manifest"],
    output_dir="validation", resume=True,
), dependencies=deps)
print(json.dumps(validation, indent=2))

In [ ]:
# Each receipt is atomic. Rerun this cell after interruption to resume.
seal = json.loads(Path(validation["seal_manifest"]).read_text())
schedule = expected_causal_shards(seal)
for index, item in enumerate(schedule, start=1):
    result = run_causal_shard(argparse.Namespace(
        config=str(CONFIG), root=str(ARTIFACT_ROOT),
        prepare_manifest=prepared["prepare_manifest"],
        seal_manifest=validation["seal_manifest"],
        split=item["split"], control=item["control"],
        unit_id=item["unit_id"], member=item["member"],
        output_dir="evidence", resume=True,
    ), dependencies=deps)
    if index % 12 == 0 or index == len(schedule):
        print(f"{index}/{len(schedule)}: {result['status']}")

In [ ]:
# One-use protected evaluation. Run only after the shard cell completes.
result = evaluate_causal(argparse.Namespace(
    config=str(CONFIG), root=str(ARTIFACT_ROOT),
    prepare_manifest=prepared["prepare_manifest"],
    seal_manifest=validation["seal_manifest"],
    evidence_dir="evidence", output_dir="results",
), dependencies=deps)
print(json.dumps(result, indent=2))